# GSE25065 — preparation for external validation

This notebook prepares **GSE25065** as the completely independent external-validation dataset.

It performs only data loading, validation, patient/label alignment and checkpoint creation. It does **not** fit LASSO, select features, tune parameters or evaluate a model. All model-related decisions must already have been made using GSE25055.

## Step 1 — Imports and file paths

GSE25065 uses the same Affymetrix GPL96 platform as GSE25055. The expression Series Matrix and the already downloaded GPL96 annotation are kept as separate source files.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve
import csv
import gzip
import re

import GEOparse
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = "GSE25065"

SERIES_MATRIX_PATH = (
    DATA_DIR / "GSE25065_series_matrix.txt.gz"
)

GPL96_PATH = DATA_DIR / "GPL96.txt"

CHECKPOINT_PATH = (
    DATA_DIR / "GSE25065_pre_lasso.joblib"
)

SERIES_MATRIX_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE25nnn/GSE25065/matrix/"
    "GSE25065_series_matrix.txt.gz"
)

print("Project directory:", PROJECT_DIR.resolve())
print("Series Matrix:", SERIES_MATRIX_PATH.resolve())
print("GPL96 annotation:", GPL96_PATH.resolve())

Project directory: D:\diplom-project
Series Matrix: D:\diplom-project\data\GSE25065_series_matrix.txt.gz
GPL96 annotation: D:\diplom-project\data\GPL96.txt


## Step 2 — Download the Series Matrix if necessary

The source file is downloaded only when it is not already present. The GPL96 annotation is reused because both datasets were measured on the same platform.

In [ ]:
if not SERIES_MATRIX_PATH.exists():
    print("Downloading GSE25065 Series Matrix...")

    urlretrieve(
        SERIES_MATRIX_URL,
        SERIES_MATRIX_PATH
    )

    print("Download completed.")
else:
    print("GSE25065 Series Matrix already exists.")


if not GPL96_PATH.exists():
    raise FileNotFoundError(
        f"GPL96 annotation not found: {GPL96_PATH.resolve()}"
    )


print("GSE25065 exists:", SERIES_MATRIX_PATH.exists())
print("GPL96 exists:", GPL96_PATH.exists())

## Step 3 — Load the expression table

In the GEO Series Matrix, rows are probe sets and columns are patients. Metadata lines begin with `!` and are skipped here because they are parsed separately later.

In [ ]:
expression_data = pd.read_csv(
    SERIES_MATRIX_PATH,
    sep="\t",
    comment="!",
    compression="gzip",
    index_col=0
)

print("Original expression shape:", expression_data.shape)

display(
    expression_data.iloc[:10, :5]
)

## Step 4 — Clean identifiers and transpose the matrix

Machine-learning algorithms expect one patient per row and one feature per column. Probe IDs remain the model features.

In [ ]:
expression_data.index = (
    expression_data.index
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

expression_data.columns = (
    expression_data.columns
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

expression_data.index.name = "PROBEID"

X_all = expression_data.T.copy().astype(float)

X_all.index.name = "sample_id"
X_all.columns.name = "PROBEID"

print("Complete patient-by-probe matrix:", X_all.shape)
print("Patients:", X_all.shape[0])
print("Probe features:", X_all.shape[1])

display(
    X_all.iloc[:5, :10]
)

## Step 5 — Validate the expression matrix

GSE25065 is expected to contain 198 samples measured over the same 22,283 GPL96 probe sets. These checks detect orientation errors, duplicate identifiers, missing values and non-finite values before labels are added.

In [ ]:
EXPECTED_PATIENTS = 198
EXPECTED_PROBES = 22283

print("Duplicate patients:", X_all.index.duplicated().sum())
print("Duplicate probes:", X_all.columns.duplicated().sum())
print("Missing values:", X_all.isna().sum().sum())
print("All values finite:", np.isfinite(X_all.to_numpy()).all())

assert X_all.shape == (
    EXPECTED_PATIENTS,
    EXPECTED_PROBES
)

assert X_all.index.is_unique
assert X_all.columns.is_unique
assert X_all.isna().sum().sum() == 0
assert np.isfinite(X_all.to_numpy()).all()

print("Expression validation passed.")

## Step 6 — Load and align GPL96 annotation

The annotation is used for interpretation only. The actual model features remain the probe-set identifiers.

In [ ]:
gpl96 = GEOparse.get_GEO(
    filepath=str(GPL96_PATH),
    silent=True
)

annotation_raw = gpl96.table.copy()

required_annotation_columns = [
    "ID",
    "Gene Symbol",
    "Gene Title",
    "ENTREZ_GENE_ID"
]

missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in annotation_raw.columns
]

if missing_annotation_columns:
    raise ValueError(
        "Missing GPL96 columns: "
        f"{missing_annotation_columns}"
    )

annotation = annotation_raw[
    required_annotation_columns
].copy()

annotation = annotation.rename(
    columns={
        "ID": "PROBEID",
        "Gene Symbol": "SYMBOL",
        "Gene Title": "GENENAME",
        "ENTREZ_GENE_ID": "ENTREZID"
    }
)

annotation["PROBEID"] = (
    annotation["PROBEID"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

gene_annotation = (
    annotation
    .drop_duplicates(subset="PROBEID")
    .set_index("PROBEID")
    .reindex(X_all.columns)
    .reset_index()
)

assert len(gene_annotation) == X_all.shape[1]
assert (
    gene_annotation["PROBEID"].tolist()
    == X_all.columns.tolist()
)

print("Annotation aligned with expression features:", len(gene_annotation))

display(gene_annotation.head(20))

In [ ]:
symbol_text = (
    gene_annotation["SYMBOL"]
    .fillna("")
    .astype(str)
    .str.strip()
)

missing_symbol_mask = symbol_text.eq("")

multiple_gene_mask = symbol_text.str.contains(
    "///",
    regex=False
)

gene_annotation["NUMBER_OF_GENES"] = np.where(
    missing_symbol_mask,
    0,
    symbol_text.str.count(r"\s*///\s*") + 1
)

gene_annotation["MAPPING_STATUS"] = np.select(
    [
        missing_symbol_mask,
        multiple_gene_mask
    ],
    [
        "No gene symbol",
        "Multiple genes"
    ],
    default="Unique"
)

display(
    gene_annotation["MAPPING_STATUS"]
    .value_counts()
    .rename_axis("Mapping status")
    .reset_index(name="Number of probes")
)

## Step 7 — Extract pCR/RD labels from GEO metadata

The patient accessions and treatment-response values are read from the same Series Matrix. They are not inferred from row position.

In [ ]:
with gzip.open(
    SERIES_MATRIX_PATH,
    mode="rt",
    encoding="utf-8"
) as file:
    metadata_lines = file.readlines()


def parse_geo_metadata_line(line):
    parsed = next(
        csv.reader(
            [line],
            delimiter="\t",
            quotechar='"'
        )
    )

    return parsed[1:]


sample_lines = [
    line
    for line in metadata_lines
    if line.startswith("!Sample_geo_accession")
]

response_lines = [
    line
    for line in metadata_lines
    if (
        line.startswith("!Sample_characteristics")
        and "pathologic_response_pcr_rd" in line.lower()
    )
]

print("Sample-ID lines found:", len(sample_lines))
print("Response lines found:", len(response_lines))

if len(sample_lines) != 1:
    raise ValueError(
        "Expected exactly one Sample_geo_accession line."
    )

if len(response_lines) != 1:
    available_characteristics = [
        line.split("\t", 1)[0]
        for line in metadata_lines
        if line.startswith("!Sample_characteristics")
    ]

    print(
        "Available Sample_characteristics lines:",
        available_characteristics
    )

    raise ValueError(
        "Could not identify exactly one "
        "pathologic_response_pcr_rd metadata line."
    )


metadata_sample_ids = parse_geo_metadata_line(
    sample_lines[0]
)

response_values = parse_geo_metadata_line(
    response_lines[0]
)

response_values = [
    re.sub(
        r"^pathologic_response_pcr_rd:\s*",
        "",
        value,
        flags=re.IGNORECASE
    ).strip()
    for value in response_values
]

print("Patient IDs:", len(metadata_sample_ids))
print("Response values:", len(response_values))
print("Unique responses:", sorted(set(response_values)))

assert len(metadata_sample_ids) == X_all.shape[0]
assert len(response_values) == X_all.shape[0]

## Step 8 — Build and align the clinical table

`RD` is encoded as class 0 and `pCR` as class 1. Unknown outcomes are retained for documentation and excluded only from supervised evaluation.

In [ ]:
clinical_outcome = pd.DataFrame({
    "Patient": metadata_sample_ids,
    "Treatment_Response": response_values
})

clinical_outcome["Treatment_Response"] = (
    clinical_outcome["Treatment_Response"]
    .astype(str)
    .str.strip()
)

clinical_outcome["Response_Label"] = (
    clinical_outcome["Treatment_Response"]
    .str.upper()
    .map({
        "PCR": 1,
        "RD": 0
    })
)

print("Clinical table shape:", clinical_outcome.shape)

print("Response distribution:")
print(
    clinical_outcome["Treatment_Response"]
    .value_counts(dropna=False)
)

print("Numeric label distribution:")
print(
    clinical_outcome["Response_Label"]
    .value_counts(dropna=False)
    .sort_index()
)

display(clinical_outcome.head(20))

In [ ]:
assert clinical_outcome["Patient"].is_unique

clinical_by_patient = (
    clinical_outcome
    .set_index("Patient")
)

missing_clinical_rows = X_all.index.difference(
    clinical_by_patient.index
)

extra_clinical_rows = clinical_by_patient.index.difference(
    X_all.index
)

print(
    "Expression patients without clinical metadata:",
    len(missing_clinical_rows)
)

print(
    "Clinical patients without expression data:",
    len(extra_clinical_rows)
)

assert len(missing_clinical_rows) == 0
assert len(extra_clinical_rows) == 0

clinical_aligned = clinical_by_patient.reindex(
    X_all.index
)

known_response_mask = (
    clinical_aligned["Response_Label"].notna()
)

excluded_patients = clinical_aligned[
    ~known_response_mask
].copy()

X = X_all.loc[
    known_response_mask
].copy()

y = (
    clinical_aligned
    .loc[known_response_mask, "Response_Label"]
    .astype("int64")
)

y.name = "Response_Label"

print("Excluded patients:", len(excluded_patients))
display(excluded_patients)

print("Complete expression matrix:", X_all.shape)
print("External-validation matrix:", X.shape)
print("Target vector:", y.shape)

## Step 9 — Final validation before checkpoint creation

These checks verify alignment and data integrity. The observed class counts are printed rather than forced to match GSE25055.

In [ ]:
print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

print("Class distribution:")
print(y.value_counts().sort_index())

print("Patient order aligned:", X.index.equals(y.index))
print("Missing expression values:", X.isna().sum().sum())
print("Missing response labels:", y.isna().sum())
print("Duplicate patients:", X.index.duplicated().sum())
print("Duplicate probes:", X.columns.duplicated().sum())

assert len(X) == len(y)
assert X.index.equals(y.index)
assert X.index.is_unique
assert X.columns.is_unique
assert X.shape[1] == EXPECTED_PROBES
assert X.isna().sum().sum() == 0
assert y.isna().sum() == 0
assert set(y.unique()) == {0, 1}

print("GSE25065 is ready for external validation.")

## Step 10 — Check feature compatibility and expression scale

This is a schema and scale validation, not model selection. No GSE25065 outcome is used to change preprocessing or parameters.

In [ ]:
TRAIN_CHECKPOINT_PATH = (
    DATA_DIR / "GSE25055_pre_lasso.joblib"
)

if not TRAIN_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "GSE25055 checkpoint is required for compatibility checks."
    )

training_checkpoint = joblib.load(
    TRAIN_CHECKPOINT_PATH
)

X_training = training_checkpoint["X"]

missing_training_probes = X_training.columns.difference(
    X.columns
)

extra_external_probes = X.columns.difference(
    X_training.columns
)

print("Training probes missing from GSE25065:", len(missing_training_probes))
print("Extra GSE25065 probes:", len(extra_external_probes))
print("Identical probe order:", X_training.columns.equals(X.columns))

assert len(missing_training_probes) == 0
assert len(extra_external_probes) == 0

# Reorder explicitly, even when the existing order is already correct.
X = X.reindex(columns=X_training.columns)

assert X_training.columns.equals(X.columns)

percentiles = [0, 1, 25, 50, 75, 99, 100]

scale_comparison = pd.DataFrame({
    "Percentile": percentiles,
    "GSE25055": np.percentile(
        X_training.to_numpy(dtype=float),
        percentiles
    ),
    "GSE25065": np.percentile(
        X.to_numpy(dtype=float),
        percentiles
    )
})

display(scale_comparison)

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    X_training.to_numpy(dtype=float).ravel(),
    bins=60,
    alpha=0.5,
    label="GSE25055"
)

plt.hist(
    X.to_numpy(dtype=float).ravel(),
    bins=60,
    alpha=0.5,
    label="GSE25065"
)

plt.xlabel("Expression value")
plt.ylabel("Frequency")
plt.title("Expression-scale comparison")
plt.legend()
plt.tight_layout()
plt.show()

## Step 11 — Save the GSE25065 checkpoint

The checkpoint stores prepared external-validation data. It does not contain fitted preprocessing, selected probes or model parameters.

In [ ]:
checkpoint = {
    "dataset_id": DATASET_ID,
    "checkpoint_version": 1,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "role": "external_validation_only",
    "label_mapping": {
        "RD": 0,
        "pCR": 1
    },
    "number_of_original_patients": len(X_all),
    "number_of_validation_patients": len(X),
    "number_of_excluded_patients": len(excluded_patients),
    "X": X,
    "y": y,
    "gene_annotation": gene_annotation,
    "clinical_outcome": clinical_outcome,
    "excluded_patients": excluded_patients,
    "scale_comparison": scale_comparison
}

joblib.dump(
    checkpoint,
    CHECKPOINT_PATH,
    compress=3
)

print("Checkpoint saved:")
print(CHECKPOINT_PATH.resolve())

print(
    "File size in MB:",
    round(
        CHECKPOINT_PATH.stat().st_size
        / (1024 ** 2),
        2
    )
)

## Step 12 — Reload and verify the checkpoint

Reloading confirms that the saved artifact contains the expected aligned data and metadata.

In [ ]:
saved_checkpoint = joblib.load(
    CHECKPOINT_PATH
)

saved_X = saved_checkpoint["X"]
saved_y = saved_checkpoint["y"]

assert saved_checkpoint["dataset_id"] == "GSE25065"
assert saved_checkpoint["role"] == "external_validation_only"
assert saved_X.equals(X)
assert saved_y.equals(y)
assert saved_X.index.equals(saved_y.index)
assert X_training.columns.equals(saved_X.columns)

print("Checkpoint verification passed.")
print("Saved X:", saved_X.shape)
print("Saved y:", saved_y.shape)
print("Saved class distribution:")
print(saved_y.value_counts().sort_index())

## Result

`GSE25065_pre_lasso.joblib` is now a validated external-validation checkpoint.

The next notebook must load the already locked model package trained on GSE25055, select only its stored probe IDs from GSE25065, generate predictions once and save the final external-validation metrics. No LASSO fitting, parameter selection or threshold tuning may be performed on GSE25065.